In [ ]:
import random
import time

import numpy as np
from numba import cuda


# ------------------------------------------------------------
# CUDA kernel: один поток проверяет одну пару
# ------------------------------------------------------------
@cuda.jit
def gpu_check_patterns(data, patterns, starts, sizes, marks, data_size, pattern_count):
    tid = cuda.grid(1)

    total_jobs = data_size * pattern_count

    if tid >= total_jobs:
        return

    pattern_id = tid // data_size
    pos = tid % data_size

    pattern_len = sizes[pattern_id]

    if pos + pattern_len > data_size:
        return

    begin = starts[pattern_id]

    for i in range(pattern_len):
        if data[pos + i] != patterns[begin + i]:
            return

    marks[tid] = 1


# ------------------------------------------------------------
# Проверка CUDA
# ------------------------------------------------------------
def init_cuda():
    if not cuda.is_available():
        raise RuntimeError("CUDA не обнаружена")

    device = cuda.get_current_device()

    print()
    print("Поиск набора байтовых шаблонов")
    print("Режим выполнения: CPU + CUDA")
    print(f"CUDA-устройство: {device.name}")
    print("-" * 50)


# ------------------------------------------------------------
# Создание тестовых данных
# ------------------------------------------------------------
def make_input_data(text_len, template_count, min_size, max_size):
    source = np.random.randint(0, 256, text_len, dtype=np.uint8)

    templates = []

    for _ in range(template_count):
        size = random.randint(min_size, max_size)
        left = random.randint(0, text_len - size)

        piece = source[left:left + size].copy()
        templates.append(piece)

    random.shuffle(templates)

    return source, templates


# ------------------------------------------------------------
# Упаковка шаблонов в один массив
# ------------------------------------------------------------
def pack_templates(templates):
    sizes = np.array([len(t) for t in templates], dtype=np.int32)

    starts = np.zeros(len(templates) + 1, dtype=np.int32)

    for i in range(len(templates)):
        starts[i + 1] = starts[i] + sizes[i]

    packed = np.zeros(starts[-1], dtype=np.uint8)

    for i, t in enumerate(templates):
        packed[starts[i]:starts[i + 1]] = t

    return packed, starts, sizes


# ------------------------------------------------------------
# CPU-поиск
# ------------------------------------------------------------
def find_on_cpu(source, templates):
    found = []

    for template_id, template in enumerate(templates):
        size = len(template)

        limit = len(source) - size + 1

        for pos in range(limit):
            equal = True

            for j in range(size):
                if source[pos + j] != template[j]:
                    equal = False
                    break

            if equal:
                found.append((template_id, pos))

    return found


# ------------------------------------------------------------
# GPU-поиск
# ------------------------------------------------------------
def find_on_gpu(source, packed, starts, sizes, template_count):
    text_len = len(source)
    job_count = text_len * template_count

    result_flags = np.zeros(job_count, dtype=np.uint8)

    d_source = cuda.to_device(source)
    d_packed = cuda.to_device(packed)
    d_starts = cuda.to_device(starts)
    d_sizes = cuda.to_device(sizes)
    d_flags = cuda.to_device(result_flags)

    threads = 256
    blocks = (job_count + threads - 1) // threads

    # прогрев
    gpu_check_patterns[blocks, threads](
        d_source,
        d_packed,
        d_starts,
        d_sizes,
        d_flags,
        text_len,
        template_count
    )
    cuda.synchronize()

    # основной запуск
    d_flags = cuda.to_device(np.zeros(job_count, dtype=np.uint8))

    start = time.time()

    gpu_check_patterns[blocks, threads](
        d_source,
        d_packed,
        d_starts,
        d_sizes,
        d_flags,
        text_len,
        template_count
    )

    cuda.synchronize()

    elapsed = time.time() - start

    result_flags = d_flags.copy_to_host()

    found = []

    for template_id in range(template_count):
        base = template_id * text_len

        for pos in range(text_len):
            if result_flags[base + pos] == 1:
                found.append((template_id, pos))

    return found, elapsed


# ------------------------------------------------------------
# Вывод итогов
# ------------------------------------------------------------
def print_report(cpu_result, gpu_result, cpu_time, gpu_time):
    print()
    print("Итог сравнения")
    print("-" * 50)

    print(f"CPU: {cpu_time:.5f} сек")
    print(f"GPU: {gpu_time:.5f} сек")

    if gpu_time > 0:
        print(f"Отношение CPU/GPU: {cpu_time / gpu_time:.2f}")

    cpu_set = set(cpu_result)
    gpu_set = set(gpu_result)

    print(f"Совпадений CPU: {len(cpu_result)}")
    print(f"Совпадений GPU: {len(gpu_result)}")

    if cpu_set == gpu_set:
        print("Проверка корректности: успешно")
    else:
        print("Проверка корректности: ошибка")
        print(f"Лишние в CPU: {len(cpu_set - gpu_set)}")
        print(f"Лишние в GPU: {len(gpu_set - cpu_set)}")

    if cpu_result:
        print("Несколько найденных позиций:")
        for item in cpu_result[:5]:
            print(" ", item)


# ------------------------------------------------------------
# Главная программа
# ------------------------------------------------------------
def main():
    init_cuda()

    text_length = 50_000
    templates_number = 100
    smallest_template = 3
    biggest_template = 15

    print(f"Длина массива данных: {text_length}")
    print(f"Количество шаблонов: {templates_number}")
    print(f"Размер шаблона: от {smallest_template} до {biggest_template}")
    print("-" * 50)

    source, templates = make_input_data(
        text_length,
        templates_number,
        smallest_template,
        biggest_template
    )

    packed_templates, starts, sizes = pack_templates(templates)

    print("CPU-часть запущена")
    start = time.time()
    cpu_result = find_on_cpu(source, templates)
    cpu_time = time.time() - start
    print("CPU-часть завершена")

    print("CUDA-часть запущена")
    gpu_result, gpu_time = find_on_gpu(
        source,
        packed_templates,
        starts,
        sizes,
        templates_number
    )
    print("CUDA-часть завершена")

    print_report(cpu_result, gpu_result, cpu_time, gpu_time)


if __name__ == "__main__":
    main()


Поиск набора байтовых шаблонов
Режим выполнения: CPU + CUDA
CUDA-устройство: Tesla T4
--------------------------------------------------
Длина массива данных: 50000
Количество шаблонов: 100
Размер шаблона: от 3 до 15
--------------------------------------------------
CPU-часть запущена
CPU-часть завершена
CUDA-часть запущена
CUDA-часть завершена

Итог сравнения
--------------------------------------------------
CPU: 4.47357 сек
GPU: 0.00051 сек
Отношение CPU/GPU: 8686.82
Совпадений CPU: 100
Совпадений GPU: 100
Проверка корректности: успешно
Несколько найденных позиций:
  (0, 4460)
  (1, 39087)
  (2, 30454)
  (3, 47582)
  (4, 48644)
